# Exploring the HLS L30 T13SDA cube

This notebook opens the virtual Icechunk datacube built by this repo's pipeline: every
Harmonized Landsat Sentinel-2 **L30** granule for MGRS tile **T13SDA** (northern New
Mexico, around Taos and Española), stacked along `time`.

The cube stores **references**, not pixels. Opening it reads only Icechunk metadata; each
pixel read fetches the matching byte range from LP DAAC's original COGs. So the cost of a
query is the number of COG chunks it touches, which is why the examples below pick the
coarsest pyramid level that answers the question.

Layout: `/{tile}/{band}/multiscales/{level}/{band}` with dims `(time, y, x)` on the tile's
UTM 13N grid (EPSG:32613). Level 0 is 30 m (3660 × 3660 px, 256 px chunks); levels 1–4 are
the COG overviews (1830, 915, 458, 229 px; 128 px chunks).

**Running it**

```sh
uv run --group notebook jupyter lab notebooks/explore_t13sda.ipynb
```

* AWS credentials for the cube's bucket (e.g. `AWS_PROFILE`) and an Earthdata login in
  `~/.netrc`.
* `ACCESS = "https"` works anywhere. `"s3"` reads LP DAAC's bucket directly and is much
  faster, but only from inside `us-west-2`.

**What to expect over HTTPS from a laptop** (measured on real T13SDA granules): the
whole-archive cloud scan reads ~0.8 s per scene and the point time series ~0.7 s per scene,
so each is roughly 15 minutes for the full ~1,170-scene archive; one level-2 RGB scene is
~40 s; a summer composite at level 3 is ~9 s per scene. Run from `us-west-2` with
`ACCESS = "s3"` for interactive speed, or narrow the time range first.

In [ ]:
import os
import time
from contextlib import contextmanager

import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import virtual_tiff  # noqa: F401  registers the TIFF codecs (e.g. HorizontalDelta) zarr needs to decode chunks
import xarray as xr
import zarr
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
from pyproj import Transformer
from virtualizarr_processor.hls.credentials import (
    container_credentials,
    earthdata_token,
    virtual_chunk_container,
)

## Configuration

In [ ]:
TILE = "T13SDA"
BRANCH = "main"

# Where the cube lives (the deployed stack's ICECHUNK_* settings).
BUCKET = os.environ.get("ICECHUNK_BUCKET", "hls-virtualizarr")
PREFIX = os.environ.get("ICECHUNK_PREFIX", "hls/T13SDA/L30")
REGION = os.environ.get("ICECHUNK_REGION", "us-west-2")
# Set HLS_CUBE_LOCAL to a local repo path to read that instead of S3.
LOCAL_REPO = os.environ.get("HLS_CUBE_LOCAL")

# How COG bytes are fetched from LP DAAC: "https" anywhere, "s3" inside us-west-2.
ACCESS = os.environ.get("LPDAAC_ACCESS", "https")

# Time-series point (lon, lat): irrigated orchards on the Rio Grande at Velarde.
POINT = (-105.97, 36.17)

# HLS Fmask bits: 1 cloud, 2 adjacent to cloud/shadow, 3 cloud shadow, 4 snow/ice.
CLOUDY_BITS = 0b0000_1110
NOT_CLEAR_BITS = CLOUDY_BITS | 0b0001_0000
FMASK_FILL = 255


def load(obj, attempts=4):
    # .load() with retries: LP DAAC's https endpoint returns the odd transient 502.
    for attempt in range(1, attempts + 1):
        try:
            return obj.load()
        except Exception as err:  # icechunk raises IcechunkError for fetch failures
            if attempt == attempts or "502" not in str(err) and "503" not in str(err):
                raise
            print(f"  transient LP DAAC error, retrying ({attempt}/{attempts - 1})")
            time.sleep(2 * attempt)


@contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    print(f"{label}: {time.perf_counter() - start:.1f} s")

In [ ]:
# Light chart styling: recessive axes and grid, thin marks.
SURFACE, INK, INK_2, RULE = "#fcfcfb", "#0b0b0b", "#52514e", "#c3c2b7"
BLUE, GRAY = "#2a78d6", "#b9b8b1"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": RULE,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.titlesize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": "#e6e5e0",
        "grid.linewidth": 0.6,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "text.color": INK,
        "font.size": 10,
        "lines.linewidth": 2,
        "figure.dpi": 110,
    }
)
# One-hue sequential ramp (light -> dark) and a red <-> gray <-> blue diverging map.
SEQUENTIAL = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
DIVERGING = LinearSegmentedColormap.from_list(
    "red_gray_blue", ["#b3261e", "#e34948", "#f0efec", "#2a78d6", "#104281"]
)

## Open the cube

The repo's stored config only names the `lpdaac` virtual chunk container. This reader
supplies how to reach it (S3 or HTTPS) in memory; nothing is written back.

In [ ]:
storage = (
    icechunk.local_filesystem_storage(LOCAL_REPO)
    if LOCAL_REPO
    else icechunk.s3_storage(bucket=BUCKET, prefix=PREFIX, region=REGION, from_env=True)
)
config = icechunk.RepositoryConfig.default()
if ACCESS == "https":
    # LP DAAC's https endpoint answers bursts of concurrent range requests with
    # 502s; icechunk's default (256 in flight) trips it. S3 copes with the default.
    config.max_concurrent_requests = 16
config.set_virtual_chunk_container(
    virtual_chunk_container(
        ACCESS, token=earthdata_token() if ACCESS == "https" else None
    )
)
with timed("open repo"):
    repo = icechunk.Repository.open(
        storage,
        config=config,
        authorize_virtual_chunk_access=container_credentials(ACCESS),
    )
    session = repo.readonly_session(BRANCH)
store = session.store

for snapshot in list(repo.ancestry(branch=BRANCH))[:5]:
    print(f"{snapshot.written_at:%Y-%m-%d %H:%M}  {snapshot.id}  {snapshot.message}")

## What's in it

In [ ]:
root = zarr.open_group(store, path=TILE, mode="r")
bands = sorted(name for name, _ in root.groups())
levels = sorted(int(k) for k, _ in root[f"{bands[0]}/multiscales"].groups())

rows = []
for level in levels:
    array = root[f"B04/multiscales/{level}/B04"]
    _, ny, nx = array.shape
    rows.append(
        {
            "level": level,
            "pixels": f"{ny} x {nx}",
            "pixel size (m)": round(3660 * 30 / ny, 1),
            "chunk": f"{array.chunks[1]} x {array.chunks[2]}",
            "chunks per scene": -(-ny // array.chunks[1]) * -(-nx // array.chunks[2]),
        }
    )
print("bands:", ", ".join(bands))
pd.DataFrame(rows).set_index("level")

In [ ]:
# Per-band encoding and the zarr-conventions attributes on each band group.
encodings = {}
for band in bands:
    array = root[f"{band}/multiscales/0/{band}"]
    encodings[band] = {
        "dtype": str(array.dtype),
        "fill": array.attrs.get("_FillValue"),
        "scale": array.attrs.get("scale_factor"),
    }
display(pd.DataFrame(encodings).T)
{k: v for k, v in root["B04"].attrs.items() if k != "multiscales"} | {
    "multiscales": "... see root['B04'].attrs['multiscales']"
}

In [ ]:
def open_level(bands, level, raw=False):
    # Lazily open bands of one pyramid level as one Dataset (no pixels are read yet).
    # raw=True keeps stored integers; otherwise fill becomes NaN and scale is applied.
    return xr.merge(
        [
            xr.open_zarr(
                store,
                group=f"{TILE}/{band}/multiscales/{level}",
                consolidated=False,
                chunks=None,
                mask_and_scale=not raw,
            )
            for band in bands
        ]
    )


open_level(["B04", "B05", "Fmask"], 0)

## The time axis

In [ ]:
times = open_level(["Fmask"], 4).time.to_index()
print(f"{len(times)} scenes, {times.min():%Y-%m-%d} to {times.max():%Y-%m-%d}")

per_year = times.year.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(per_year.index, per_year.values, width=0.8, color=BLUE)
ax.set_title("L30 scenes per year, T13SDA")
ax.set_ylabel("scenes")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="x", visible=False)
ax.set_xticks(per_year.index[::2])
for year, n in per_year.items():
    if year in (per_year.idxmax(), per_year.index[-1]):
        ax.annotate(
            str(n), (year, n), ha="center", va="bottom", fontsize=9, color=INK_2
        )
plt.show()

### Query: select by date

Time selection uses ordinary xarray indexing and reads nothing until values are needed.

In [ ]:
cube = open_level(["B04", "B03", "B02"], 2)
summer = cube.sel(
    time=slice(f"{times.max().year - 1}-06-01", f"{times.max().year - 1}-08-31")
)
print(f"{summer.sizes['time']} scenes in summer {times.max().year - 1}")
nearest = cube.sel(time="2020-07-04", method="nearest")
print(
    "scene nearest 2020-07-04:",
    pd.Timestamp(nearest.time.values).strftime("%Y-%m-%d %H:%M"),
)

## Cloud cover for every scene

Fmask at level 4 is 229 × 229 px in 4 chunks per scene, so scanning the whole archive reads
4 small chunks per scene instead of 225 large ones. The overview is subsampled, so these
are estimates. A scene with no valid pixels is a time step whose granule was skipped at
ingest; it holds only fill.

In [ ]:
with timed(f"read Fmask level 4 for {len(times)} scenes"):
    fmask = load(open_level(["Fmask"], 4, raw=True).Fmask)

valid = fmask != FMASK_FILL
n_valid = valid.sum(("y", "x"))
scenes = pd.DataFrame(
    {
        "coverage": (n_valid / valid[0].size).values,
        "cloud": (((fmask & CLOUDY_BITS) > 0) & valid).sum(("y", "x")).values
        / np.maximum(n_valid.values, 1),
    },
    index=times,
)
empty = scenes.coverage == 0
scenes.loc[empty, "cloud"] = np.nan
print(f"empty (fill-only) steps: {int(empty.sum())}")
scenes.describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.scatter(scenes.index, scenes.cloud * 100, s=10, color=BLUE, linewidths=0)
ax.set_title("Cloud and shadow cover per scene (Fmask, level 4)")
ax.set_ylabel("% of valid pixels")
ax.set_ylim(-2, 102)
plt.show()

clearest = (
    scenes.dropna().sort_values(["cloud", "coverage"], ascending=[True, False]).head(5)
)
clearest.style.format({"coverage": "{:.0%}", "cloud": "{:.1%}"})

## A true-colour scene

The clearest well-covered scene from the most recent year that has one, at level 2
(915 × 915 px, ~120 m). Coverage varies with the Landsat path: some passes only clip the
tile. Reflectance is already scaled to 0–1 by the stored `scale_factor`.

In [ ]:
usable = scenes.dropna()
pool = usable[usable.coverage >= 0.9]
if pool.empty:  # no near-full scenes: take the best-covered quarter
    pool = usable[usable.coverage >= usable.coverage.quantile(0.75)]
if (pool.cloud < 0.1).any():
    pool = pool[pool.cloud < 0.1]
recent = pool[pool.index.year == pool.index.year.max()]
best = recent.cloud.idxmin()
print(
    f"{best:%Y-%m-%d}: {recent.coverage[best]:.0%} coverage, "
    f"{recent.cloud[best]:.1%} cloud"
)

with timed(f"read RGB level 2 for {best:%Y-%m-%d}"):
    scene = load(open_level(["B04", "B03", "B02"], 2).sel(time=best))

rgb = np.dstack([scene[b].values for b in ("B04", "B03", "B02")])
rgb = np.clip(np.nan_to_num(rgb) / 0.25, 0, 1) ** (1 / 1.4)

to_utm = Transformer.from_crs(4326, 32613, always_xy=True)
px, py = to_utm.transform(*POINT)
extent_km = [
    v / 1000 for v in (scene.x.min(), scene.x.max(), scene.y.min(), scene.y.max())
]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb, extent=extent_km, interpolation="nearest")
ax.scatter(
    px / 1000, py / 1000, s=60, facecolors="none", edgecolors="white", linewidths=2
)
ax.annotate(
    "time-series point",
    (px / 1000, py / 1000),
    xytext=(8, 8),
    textcoords="offset points",
    color="white",
    fontsize=9,
)
ax.set_title(f"T13SDA true colour, {best:%Y-%m-%d} (level 2)")
ax.set_xlabel("UTM 13N easting (km)")
ax.set_ylabel("northing (km)")
ax.grid(False)
plt.show()

## NDVI time series at a point

Full resolution (level 0), a 3 × 3 pixel window around `POINT`. Each scene costs one
256 × 256 chunk per band, so this reads 3 chunks per scene across the archive. Over
HTTPS from a laptop that takes a while; from `us-west-2` with `ACCESS = "s3"` it is fast.

NDVI for L30 is `(B05 − B04) / (B05 + B04)`. Observations whose centre pixel Fmask flags
cloud, adjacent cloud, shadow or snow are shown but excluded from the trend.

In [ ]:
window = slice(px - 45, px + 45), slice(py + 45, py - 45)  # y is descending
with timed(f"read B04, B05, Fmask at level 0 for {len(times)} scenes"):
    refl = load(open_level(["B04", "B05"], 0).sel(x=window[0], y=window[1]))
    flags = load(open_level(["Fmask"], 0, raw=True).Fmask.sel(x=window[0], y=window[1]))

ndvi = ((refl.B05 - refl.B04) / (refl.B05 + refl.B04)).median(("y", "x"))
centre = flags.isel(y=flags.sizes["y"] // 2, x=flags.sizes["x"] // 2)
clear = ((centre & NOT_CLEAR_BITS) == 0) & (centre != FMASK_FILL)

series = pd.DataFrame(
    {"ndvi": ndvi.values, "clear": clear.values}, index=times
).dropna()
clear_ndvi = series.ndvi[series.clear]
trend = clear_ndvi.rolling("90D", center=True, min_periods=3).median()
print(f"{len(series)} observations, {len(clear_ndvi)} clear")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6), constrained_layout=True)
flagged = series[~series.clear]
ax.scatter(
    flagged.index, flagged.ndvi, s=8, color=GRAY, linewidths=0, label="flagged by Fmask"
)
ax.scatter(
    clear_ndvi.index,
    clear_ndvi,
    s=10,
    color=BLUE,
    alpha=0.55,
    linewidths=0,
    label="clear",
)
ax.plot(trend.index, trend, color=BLUE, label="90-day median (clear)")
ax.set_title(
    f"NDVI at {POINT[1]:.2f}°N, {abs(POINT[0]):.2f}°W (3 × 3 px median, level 0)"
)
ax.set_ylabel("NDVI")
ax.legend(
    loc="upper left", bbox_to_anchor=(0, -0.1), frameon=False, ncols=3, fontsize=9
)
plt.show()

In [ ]:
by_month = clear_ndvi.groupby(clear_ndvi.index.month)
climatology = pd.DataFrame(
    {
        "median": by_month.median(),
        "q25": by_month.quantile(0.25),
        "q75": by_month.quantile(0.75),
    }
).reindex(range(1, 13))

fig, ax = plt.subplots(figsize=(7, 3.4), constrained_layout=True)
ax.fill_between(
    climatology.index,
    climatology.q25,
    climatology.q75,
    color=BLUE,
    alpha=0.15,
    linewidth=0,
    label="interquartile range",
)
ax.plot(
    climatology.index,
    climatology["median"],
    color=BLUE,
    marker="o",
    markersize=5,
    label="median",
)
ax.set_xticks(
    range(1, 13), ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
)
ax.set_title("Seasonal cycle of clear-sky NDVI at the point")
ax.set_ylabel("NDVI")
ax.legend(
    loc="upper left", bbox_to_anchor=(0, -0.12), frameon=False, ncols=2, fontsize=9
)
plt.show()

## Summer NDVI: first year vs latest year

Median clear-sky NDVI over June–August at level 3 (458 × 458 px, ~240 m, 16 chunks per
scene), for the first and the latest summer in the archive, and their difference.

In [ ]:
def summer_ndvi(year, level=3):
    period = slice(f"{year}-06-01", f"{year}-08-31")
    refl = load(open_level(["B04", "B05"], level).sel(time=period))
    flags = load(open_level(["Fmask"], level, raw=True).Fmask.sel(time=period))
    clear = ((flags & NOT_CLEAR_BITS) == 0) & (flags != FMASK_FILL)
    ndvi = (refl.B05 - refl.B04) / (refl.B05 + refl.B04)
    return ndvi.where(clear).median("time"), refl.sizes["time"]


summers = sorted({t.year for t in times if 6 <= t.month <= 8})
first, last = summers[0], summers[-1]
with timed(f"summer composites {first} and {last}"):
    (ndvi_first, n_first), (ndvi_last, n_last) = summer_ndvi(first), summer_ndvi(last)
change = ndvi_last - ndvi_first

In [ ]:
extent = [
    v / 1000 for v in (change.x.min(), change.x.max(), change.y.min(), change.y.max())
]
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for ax, data, title in [
    (axes[0], ndvi_first, f"Summer {first} ({n_first} scenes)"),
    (axes[1], ndvi_last, f"Summer {last} ({n_last} scenes)"),
]:
    im = ax.imshow(
        data, extent=extent, cmap=SEQUENTIAL, vmin=0, vmax=0.8, interpolation="nearest"
    )
    ax.set_title(f"Median NDVI, {title}")
fig.colorbar(im, ax=axes[:2], shrink=0.8, label="NDVI")

im = axes[2].imshow(
    change,
    extent=extent,
    cmap=DIVERGING,
    norm=TwoSlopeNorm(vmin=-0.3, vcenter=0, vmax=0.3),
    interpolation="nearest",
)
axes[2].set_title(f"Change, {last} minus {first}")
fig.colorbar(
    im, ax=axes[2], shrink=0.8, label="NDVI difference (red: loss, blue: gain)"
)
for ax in axes:
    ax.grid(False)
    ax.set_xlabel("easting (km)")
axes[0].set_ylabel("northing (km)")
plt.show()